# RF cavity field3d — projection-free training on Colab (G4 / A100 / H100)

Branch `claude/train-no-projection`, docs/38. Run the cells in order. Data: the TRUBA `TEST=1` set copied to Google Drive
(`<Drive>/rfcav3d/F_p2p2_ms0.10_k10_mf0.05_v3_test/h5/<family>/*.h5`). Checkpoints, `progress.json` and the log go to Drive, so a
disconnected session resumes from `last.ckpt` by re-running cells 1–4 and 7.

## 1. Machine check — GPU, CPU, RAM, disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /content | tail -1
import torch; print("torch", torch.__version__, "cuda", torch.version.cuda, "capability", torch.cuda.get_device_capability())
# needed: a GPU with real FP64 or high memory bandwidth (A100/H100/H200/RTX PRO 6000); Blackwell (capability (12, 0)) needs torch >= 2.7 / CUDA >= 12.8

## 2. Settings — edit the Drive path if your folder is elsewhere

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SET = '/content/drive/MyDrive/rfcav3d/F_p2p2_ms0.10_k10_mf0.05_v3_test'   # <- folder that contains h5/
RUNS      = DRIVE_SET + '/runs_colab'          # checkpoints + progress.json + logs (on Drive: survive a disconnect)
EXP       = 'noproj_small_v3test_colab'
EPOCHS    = 50                                  # pilot; the cosine schedule spans exactly this many epochs
BATCH     = 4                                   # = TRUBA akya 4 GPU x batch 1 (same effective batch)
LR        = 3e-4
FAMILIES  = ['elliptical', 'reentrant', 'pillbox_pipes', 'ridged_box', 'composite', 'freeform', 'hwr', 'spoke', 'dtl']  # train group
REPO, BRANCH = '/content/rf_cavity_neural_operator', 'claude/train-no-projection'
DATA, CACHE  = '/content/data/h5', '/content/ops_cache'      # local disk: fast
assert os.path.isdir(DRIVE_SET + '/h5'), 'h5/ not found under ' + DRIVE_SET
print(sorted(os.listdir(DRIVE_SET + '/h5')))

## 3. Code and environment (~3–5 min)

In [ ]:
!test -d {REPO} || git clone -q -b {BRANCH} https://github.com/KorayGokceler/rf_cavity_neural_operator.git {REPO}
!cd {REPO} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH} && git log --oneline -1
!apt-get -qq install -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1 > /dev/null
# keep Colab's own CUDA torch: requirements.txt only asks torch>=2.1, pip leaves the installed one alone
!pip -q install -r {REPO}/requirements.txt
import torch; print("torch still", torch.__version__)
!cd {REPO} && python -c "import torch, ngsolve, h5py, pytorch_lightning as pl; print('ngsolve', ngsolve.__version__, 'lightning', pl.__version__, 'cuda ok', torch.cuda.is_available())"

## 4. Copy the training families to the local disk (~10 GB, a few minutes)

In [ ]:
os.makedirs(DATA, exist_ok=True)
for fam in FAMILIES:
    src = f'{DRIVE_SET}/h5/{fam}'
    if os.path.isdir(src):
        !rsync -a --info=progress2 --exclude '*.partial' --exclude '*.corrupt-*' "{src}" {DATA}/
!du -sh {DATA}/*

## 5. GPU check: one training step on a 23.7k-tet and a 51k-tet sample (~5–10 min)
TRUBA V100 reference (docs/38 §8): 23.7k tets 4.74 GB, step 0.23 + 0.51 s; 51k tets 10.18 GB, 0.49 + 1.08 s.
If the step here is much slower than the V100 (FP64 bound), stop and use an A100/H100 runtime instead.

In [ ]:
for f, s in [('elliptical/elliptical_s00000.h5', 'sample_524388'), ('ridged_box/ridged_box_s00000.h5', 'sample_2097174')]:
    if os.path.exists(f'{DATA}/{f}'):
        print('=====', s)
        !cd {REPO} && python scripts/profile_field_step.py {DATA}/{f} --sample {s} --device cuda --threads 4 --reps 2 --grad_checkpoint --no_projection 2>&1 | grep -v Warning | tail -9

## 6. Workers
Data-loader workers build / read the operators (CPU). Workers of the train AND the val loader stay alive and each prefetches 2 batches: too many workers run the
machine out of RAM (a worker 'killed by signal: Killed'). The cell keeps ~24 GB per worker, at most 10.

In [ ]:
import psutil
ram_gb = psutil.virtual_memory().total / 2**30
# train AND val loader workers stay alive (persistent_workers) and each holds prefetch_factor=2 collated batches
# (batch 4: ~0.5-1 GB of sparse operators per sample) + ~3-5 GB while it builds a sample: ~24 GB per worker pair.
# ~5-6 workers already keep a G4 busy after the first epoch (docs/38 §10).
WORKERS = max(2, min(os.cpu_count() - 2, int(ram_gb // 24), 10))
print(f'{os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM -> {WORKERS} workers'); os.makedirs(CACHE, exist_ok=True)

## 7. Train (re-run this cell after a disconnect: it resumes from last.ckpt)
Projection-free train + val (docs/38), selection on `val/span_loss`, the test at the end is projected (+ Ritz).
All samples are kept (no `max_tets`: 96 GB). The log is also written to Drive.

In [ ]:
dirs = ','.join(f'"{DATA}/{f}"' for f in FAMILIES if glob.glob(f'{DATA}/{f}/*.h5'))
ckpt = f'{RUNS}/{EXP}/last.ckpt'
resume = f'--resume "{ckpt}"' if os.path.exists(ckpt) else ''
print('resume from', ckpt) if resume else print('fresh run')
os.makedirs(f'{RUNS}/{EXP}', exist_ok=True)
!cd {REPO} && OMP_NUM_THREADS=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py --config configs/field_3d_noproj.yaml {resume} \
   --override 'dataset.data_path=[{dirs}]' dataset.cache_dir={CACHE} training.log_dir="{RUNS}" training.exp_name={EXP} \
   training.max_epochs={EPOCHS} training.batch_size={BATCH} training.learning_rate={LR} training.num_workers={WORKERS} \
   2>&1 | tee -a "{RUNS}/{EXP}/train.log" | grep -v -E "it/s]$|Warning"

## 8. Progress (any time, also from another notebook: reads progress.json on Drive)

In [ ]:
import json
p = json.load(open(f'{RUNS}/{EXP}/progress.json'))
keys = ['train/loss', 'val/loss', 'val/span_loss', 'val/span_rel_l2', 'val/mass_diag_ratio', 'val/grad_frac_lb']
for e in p.get('history', [])[-10:]:
    print(e.get('epoch'), '  '.join(f"{k.split('/')[-1]}={e[k]:.4g}" for k in keys if isinstance(e.get(k), (int, float))))
print({k: p[k] for k in p if k not in ('history',)})

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS}/{EXP}